In [91]:
import pandas as pd
import numpy as np
import os
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer, TVAESynthesizer
from sdv.metadata import Metadata

In [92]:
# Constants
SYNTHETIC_SAMPLES = 10000
DATA_PATH = "../../../../Datasets/csv/UAVATTACK/syntheticSpoofing_gan_data.csv"
METADATA_FILE = "spoofing_metadata.json"

In [93]:
# Load the dataset
try:
    df = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    print(f"Error: The file '{DATA_PATH}' was not found.")
    print("Please make sure the path is correct.")
    exit()


In [94]:
# Data Preprocessing
if 'timestamp' in df.columns:
    df = df.drop(columns=['timestamp'])

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Encode 'label'
if 'label' in df.columns and (df['label'].dtype == object or df['label'].dtype.name == 'category'):
    from sklearn.preprocessing import LabelEncoder

    le = LabelEncoder()
    df['label'] = le.fit_transform(df['label'])

In [95]:
# Load or create metadata
if os.path.exists(METADATA_FILE):
    metadata = Metadata.load_from_json(METADATA_FILE)
    print(f"Loaded metadata from {METADATA_FILE}")
else:
    metadata = Metadata()
    metadata.detect_table_from_dataframe(
        data=df,
        table_name='spoofing_data'
    )
    metadata.save_to_json(METADATA_FILE)
    print(f"Created and saved new metadata to {METADATA_FILE}")

Loaded metadata from spoofing_metadata.json


C:\Users\antho\miniforge3\envs\CTGAN\Lib\site-packages\sdv\metadata\metadata.py:37: UserWarning: You are loading an older SingleTableMetadata object. This will be converted into the new Metadata object with a placeholder table name ('table'). Please save this new object for future usage.
  warnings.warn(


In [96]:
# Synthesizer configurations
# If you don't have a GPU available, it will automatically use CPU.
synthesizers = [
    {
        # Uses GAN-based, deep learning methods to train a model and generate synthetic data.
        'name': 'CTGAN',
        'class': CTGANSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 4000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../../Datasets/csv/UAVATTACK/spoofing-CTGAN-synthetic.csv"
    },
    {
        # Uses classic, statistical methods to train a model and generate synthetic data
        'name': 'GaussianCopula',
        'class': GaussianCopulaSynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
        },
        'output': "../../../../Datasets/csv/UAVATTACK/spoofing-GaussianCopula-synthetic.csv"
    },
    {
        # Uses a variational autoencoder (VAE)-based, neural network techniques to train a model and generate synthetic data.
        'name': 'TVAE',
        'class': TVAESynthesizer,
        'params': {
            'enforce_min_max_values': True,
            'enforce_rounding': False,
            'epochs': 2000,
            'verbose': True,
            'cuda': True
        },
        'output': "../../../../Datasets/csv/UAVATTACK/spoofing-TVAE-synthetic.csv"
    }
]

In [97]:
# Fit synthesizers and generate synthetic data
# This will take some time.
for synth in synthesizers:
    print(f"\nProcessing {synth['name']}...")
    SynthClass = synth['class']
    if synth['name'] == 'GaussianCopula':
        synthesizer = SynthClass(metadata)
    else:
        synthesizer = SynthClass(metadata, **synth['params'])
    synthesizer.fit(df)
    synthetic_data = synthesizer.sample(num_rows=SYNTHETIC_SAMPLES)
    print(f"Synthetic data preview for {synth['name']}:")
    print(synthetic_data.head())
    synthetic_data.to_csv(synth['output'], index=False)
    print(f"Synthetic dataset saved to {synth['output']}")


Processing CTGAN...


Gen. (-0.01) | Discrim. (-0.14): 100%|██████████| 4000/4000 [1:06:29<00:00,  1.00it/s]


Synthetic data preview for CTGAN:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.374048 -0.005933 -0.094064  0.926873          0.999995     -4.610000e-10   
1 -0.374111  0.001033 -0.007380  0.926533          0.999995     -4.610000e-10   
2 -0.372521  0.001709 -0.016442  0.931454          0.999995     -4.610000e-10   
3 -0.373524 -0.003551 -0.013584  0.926075          0.999995     -4.610000e-10   
4 -0.379259 -0.002958 -0.016073  0.930691          0.999995     -4.610000e-10   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0      1.580000e-09          0.003237                   3  36.204953  ...   
1      1.580000e-09          0.003237                   3  36.204953  ...   
2      1.580000e-09          0.003237                   3  36.204953  ...   
3      1.580000e-09          0.003237                   3  36.204953  ...   
4      1.580000e-09          0.003237                   3  36.204953  ...   

   v_z_valid  xy

Loss: -258.306: 100%|██████████| 2000/2000 [13:25<00:00,  2.48it/s]


Synthetic data preview for TVAE:
       q_0_      q_1_      q_2_      q_3_  delta_q_reset_0_  delta_q_reset_1_  \
0 -0.375457 -0.002494 -0.008312  0.928514          0.999995     -4.610000e-10   
1 -0.372268  0.001642  0.043633  0.927189          0.999995     -4.610000e-10   
2 -0.370857 -0.002415  0.006793  0.929435          0.999995     -4.610000e-10   
3 -0.371416 -0.022469 -0.010536  0.924859          0.999995     -4.610000e-10   
4 -0.375806 -0.006742 -0.004225  0.927558          0.999995     -4.610000e-10   

   delta_q_reset_2_  delta_q_reset_3_  quat_reset_counter      lat_x  ...  \
0      1.580000e-09          0.003237                   3  36.204811  ...   
1      1.580000e-09          0.003237                   3  36.204811  ...   
2      1.580000e-09          0.003237                   3  36.204811  ...   
3      1.580000e-09          0.003237                   3  36.204811  ...   
4      1.580000e-09          0.003237                   3  36.204811  ...   

   v_z_valid  xy_